In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential
from tensorflow.keras.applications import VGG19
from tensorflow.keras.layers import Dense, Flatten, Dropout, Resizing

In [ ]:
# Load CSV files
train = pd.read_csv("mnist_train.csv")
test = pd.read_csv("mnist_test.csv")

# Separate labels and pixel values
ytrain = train.iloc[:, 0].values
xtrain = train.iloc[:, 1:].values

ytest = test.iloc[:, 0].values
xtest = test.iloc[:, 1:].values

In [ ]:
# Reshape images and scale pixel values
xtrain = xtrain.reshape(-1, 28, 28, 1) / 255.0
xtest = xtest.reshape(-1, 28, 28, 1) / 255.0

# Convert grayscale images to 3-channel images
xtrain = np.repeat(xtrain, 3, axis=-1)
xtest = np.repeat(xtest, 3, axis=-1)


In [ ]:
base_model = VGG19(
    weights="imagenet",
    include_top=False,
    input_shape=(32, 32, 3)
)
base_model.trainable = False


# Add digit-classification layers
model = Sequential([
    Resizing(32, 32, input_shape=(28, 28, 3)),
    base_model,
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.2),
    Dense(10, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
# Train and evaluate
history = model.fit(
    xtrain, ytrain,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

loss, accuracy = model.evaluate(xtest, ytest)
print(f"Test accuracy: {accuracy * 100:.2f}%")

In [ ]:
# Predict one test image
i = 1
prediction = model.predict(xtest[i:i+1], verbose=0).argmax()
print("Predicted:", prediction, "Actual:", ytest[i])

plt.imshow(xtest[i])
plt.axis("off")
plt.show()


# Plot accuracy
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.legend()
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.show()